# **Импорт библиотек**

In [12]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset, random_split
from torchvision import datasets, transforms

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
import torch

%matplotlib inline

# **Загрузка и подготовка данных**

In [13]:


# Преобразование: ToTensor -> [0,1], затем flatten из (1,28,28) в (784)
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Lambda(lambda x: x.view(-1))  # flatten
])

train_dataset = datasets.FashionMNIST(root='./data', train=True, download=True, transform=transform)
test_dataset = datasets.FashionMNIST(root='./data', train=False, download=True, transform=transform)

# Разделение на 50000 / 10000
train_size = 50000
val_size = len(train_dataset) - train_size
train_subset, val_subset = random_split(train_dataset, [train_size, val_size])

batch_size = 128
train_loader = DataLoader(train_subset, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_subset, batch_size=batch_size, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

print(f"Обучающих примеров: {len(train_subset)}")
print(f"Проверочных примеров: {len(val_subset)}")
print(f"Тестовых примеров: {len(test_dataset)}")

Обучающих примеров: 50000
Проверочных примеров: 10000
Тестовых примеров: 10000


# **Функция обучения модели**

In [14]:
def train_model(model, name, epochs=20, lr=0.001, optimizer_class=optim.Adam, **opt_kwargs):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model.to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = optimizer_class(model.parameters(), lr=lr, **opt_kwargs)

    train_losses, val_losses = [], []
    train_accs, val_accs = [], []

    for epoch in range(epochs):
        model.train()
        train_loss = 0.0
        correct_train = 0
        total_train = 0

        for inputs, labels in train_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            train_loss += loss.item() * inputs.size(0)
            _, predicted = torch.max(outputs, 1)
            total_train += labels.size(0)
            correct_train += (predicted == labels).sum().item()

        avg_train_loss = train_loss / len(train_loader.dataset)
        train_acc = correct_train / total_train
        train_losses.append(avg_train_loss)
        train_accs.append(train_acc)

        # Валидация
        model.eval()
        val_loss = 0.0
        correct_val = 0
        total_val = 0
        with torch.no_grad():
            for inputs, labels in val_loader:
                inputs, labels = inputs.to(device), labels.to(device)
                outputs = model(inputs)
                loss = criterion(outputs, labels)
                val_loss += loss.item() * inputs.size(0)
                _, predicted = torch.max(outputs, 1)
                total_val += labels.size(0)
                correct_val += (predicted == labels).sum().item()

        avg_val_loss = val_loss / len(val_loader.dataset)
        val_acc = correct_val / total_val
        val_losses.append(avg_val_loss)
        val_accs.append(val_acc)

        if (epoch+1) % 5 == 0:
            print(f"{name} | Epoch {epoch+1}/{epochs} | Train Acc: {train_acc:.4f} | Val Acc: {val_acc:.4f}")

    # Оценка на тесте
    model.eval()
    correct_test = 0
    total_test = 0
    with torch.no_grad():
        for inputs, labels in test_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            _, predicted = torch.max(outputs, 1)
            total_test += labels.size(0)
            correct_test += (predicted == labels).sum().item()
    test_acc = correct_test / total_test

    print(f"{name} | Test Acc: {test_acc:.4f}")
    return model, train_losses, val_losses, train_accs, val_accs, test_acc

# **Создание 9 моделей**

In [15]:
# Модель 1: один слой 128
class Model1(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc = nn.Linear(784, 128)
        self.out = nn.Linear(128, 10)

    def forward(self, x):
        x = F.relu(self.fc(x))
        return self.out(x)

# Модель 2: два слоя 256, 128
class Model2(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc1 = nn.Linear(784, 256)
        self.fc2 = nn.Linear(256, 128)
        self.out = nn.Linear(128, 10)

    def forward(self, x):
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        return self.out(x)

# Модель 3: три слоя 512, 256, 128
class Model3(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc1 = nn.Linear(784, 512)
        self.fc2 = nn.Linear(512, 256)
        self.fc3 = nn.Linear(256, 128)
        self.out = nn.Linear(128, 10)

    def forward(self, x):
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        x = F.relu(self.fc3(x))
        return self.out(x)

# Модель 4: с Dropout (256, Dropout 0.5, 128)
class Model4(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc1 = nn.Linear(784, 256)
        self.dropout = nn.Dropout(0.5)
        self.fc2 = nn.Linear(256, 128)
        self.out = nn.Linear(128, 10)

    def forward(self, x):
        x = F.relu(self.fc1(x))
        x = self.dropout(x)
        x = F.relu(self.fc2(x))
        return self.out(x)

# Модель 5: с BatchNorm (256, BatchNorm, ReLU, 128)
class Model5(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc1 = nn.Linear(784, 256)
        self.bn = nn.BatchNorm1d(256)
        self.fc2 = nn.Linear(256, 128)
        self.out = nn.Linear(128, 10)

    def forward(self, x):
        x = self.fc1(x)
        x = self.bn(x)
        x = F.relu(x)
        x = F.relu(self.fc2(x))
        return self.out(x)

# Модель 6: tanh вместо ReLU (256, tanh, 128, tanh)
class Model6(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc1 = nn.Linear(784, 256)
        self.fc2 = nn.Linear(256, 128)
        self.out = nn.Linear(128, 10)

    def forward(self, x):
        x = torch.tanh(self.fc1(x))
        x = torch.tanh(self.fc2(x))
        return self.out(x)

# Модель 7: маленькая (64, 32)
class Model7(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc1 = nn.Linear(784, 64)
        self.fc2 = nn.Linear(64, 32)
        self.out = nn.Linear(32, 10)

    def forward(self, x):
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        return self.out(x)

# Модель 8: большая (1024, 512)
class Model8(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc1 = nn.Linear(784, 1024)
        self.fc2 = nn.Linear(1024, 512)
        self.out = nn.Linear(512, 10)

    def forward(self, x):
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        return self.out(x)

# Модель 9: комбинированная (512, Dropout0.3, 256, BatchNorm, 128, SGD)
class Model9(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc1 = nn.Linear(784, 512)
        self.dropout = nn.Dropout(0.3)
        self.fc2 = nn.Linear(512, 256)
        self.bn = nn.BatchNorm1d(256)
        self.fc3 = nn.Linear(256, 128)
        self.out = nn.Linear(128, 10)

    def forward(self, x):
        x = F.relu(self.fc1(x))
        x = self.dropout(x)
        x = F.relu(self.fc2(x))
        x = self.bn(x)
        x = F.relu(self.fc3(x))
        return self.out(x)

# Список моделей и их названий
models = [
    Model1(), Model2(), Model3(), Model4(), Model5(),
    Model6(), Model7(), Model8(), Model9()
]
names = [
    '1_layer_128', '2_layers_256_128', '3_layers_512_256_128',
    'dropout_0.5', 'batchnorm', 'tanh', 'small_64_32',
    'large_1024_512', 'combined_SGD'
]

# **Обучение всех моделей и сбор результатов**

In [16]:
results = []
histories = []  # сохраним все данные обучения для графиков

for model, name in zip(models, names):
    # Для модели 9 используем SGD, для остальных Adam
    if name == 'combined_SGD':
        _, train_losses, val_losses, train_accs, val_accs, test_acc = train_model(
            model, name, epochs=20, lr=0.01, optimizer_class=optim.SGD, momentum=0.9
        )
    else:
        _, train_losses, val_losses, train_accs, val_accs, test_acc = train_model(
            model, name, epochs=20, lr=0.001, optimizer_class=optim.Adam
        )

    results.append({
        'Модель': name,
        'Точность на проверке': val_accs[-1],  # последнее значение на валидации
        'Точность на тесте': test_acc
    })
    histories.append((train_losses, val_losses, train_accs, val_accs))

1_layer_128 | Epoch 5/20 | Train Acc: 0.8773 | Val Acc: 0.8716
1_layer_128 | Epoch 10/20 | Train Acc: 0.8965 | Val Acc: 0.8750
1_layer_128 | Epoch 15/20 | Train Acc: 0.9076 | Val Acc: 0.8855


KeyboardInterrupt: 

# **Сравнительная таблица**

In [ ]:
df_results = pd.DataFrame(results)
print("Сравнительная таблица результатов:")
print(df_results.to_string(index=False))